# Vulnerability Classification and Action-Selection Model Training

This notebook trains a vulnerability classification and action-selection model using LoRA fine-tuning on a JSONL dataset of security vulnerability examples.

## Overview

The model will be trained to:
- Classify different types of web security vulnerabilities (SQL injection, XSS, path traversal, etc.)
- Select appropriate patches/fixes for identified vulnerabilities
- Make workflow decisions for security analysis tasks

Dataset: `dataset_300_final_winning.jsonl` - Contains 300+ examples across multiple categories.

## 1. Setup and Dependencies

Install required packages and set up the environment for training.

In [ ]:
# Install required packages
!pip install torch transformers datasets peft accelerate bitsandbytes trl

# Import libraries
import torch
import json
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import os

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Load Dataset

Load the JSONL dataset containing vulnerability classification examples.

In [ ]:
# Load the JSONL dataset
dataset_path = "dataset_300_final_winning.jsonl"

def load_jsonl_dataset(file_path):
    """Load JSONL dataset into a list of dictionaries."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Error parsing line: {e}")
                    continue
    return data

# Load the dataset
raw_data = load_jsonl_dataset(dataset_path)
print(f"Loaded {len(raw_data)} examples from {dataset_path}")

# Convert to pandas DataFrame for inspection
df = pd.DataFrame(raw_data)
print("\nDataset overview:")
print(f"Columns: {list(df.columns)}")
print(f"Categories: {df['category'].value_counts().to_dict()}")
print(f"\nSample example:")
print(json.dumps(raw_data[0], indent=2))

## 3. Data Preprocessing

Convert dataset examples into text prompts suitable for instruction tuning.

In [ ]:
def format_input_field(input_field):
    """Format the input field which can be a string, dict, or other type."""
    if isinstance(input_field, str):
        return input_field
    elif isinstance(input_field, dict):
        # Handle structured input (code, scanner_hint, options, etc.)
        formatted_parts = []
        if 'code' in input_field:
            formatted_parts.append(f"Code: {input_field['code']}")
        if 'scanner_hint' in input_field:
            formatted_parts.append(f"Scanner hint: {input_field['scanner_hint']}")
        if 'options' in input_field and isinstance(input_field['options'], list):
            formatted_parts.append(f"Options: {', '.join(input_field['options'])}")
        if 'state' in input_field:
            formatted_parts.append(f"State: {input_field['state']}")
        if 'hint' in input_field:
            formatted_parts.append(f"Hint: {input_field['hint']}")
        return '\n'.join(formatted_parts)
    else:
        return str(input_field)

def create_prompt(example):
    """Convert a dataset example into a training prompt."""
    instruction = example.get('instruction', '')
    input_text = format_input_field(example.get('input', ''))
    output = example.get('output', '')

    # Create the full prompt
    prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput: {output}"

    return {
        'text': prompt,
        'instruction': instruction,
        'input': input_text,
        'output': output,
        'category': example.get('category', 'unknown')
    }

# Preprocess all examples
processed_data = [create_prompt(example) for example in raw_data]

print(f"Processed {len(processed_data)} examples")
print("\nSample processed example:")
print(processed_data[0]['text'][:500] + "..." if len(processed_data[0]['text']) > 500 else processed_data[0]['text'])

# Convert to Hugging Face Dataset
dataset = Dataset.from_list(processed_data)
print(f"\nDataset columns: {dataset.column_names}")
print(f"Dataset size: {len(dataset)}")

## 4. Train/Validation Split

Split the dataset into training and validation sets.

In [ ]:
# Split dataset into train and validation sets
train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
val_dataset = train_test_split['test']

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

# Show distribution by category
print("\nTraining set category distribution:")
print(train_dataset.to_pandas()['category'].value_counts())
print("\nValidation set category distribution:")
print(val_dataset.to_pandas()['category'].value_counts())

## 5. Model Loading

Load a small instruction-tuned model with 4-bit quantization for efficient training.

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
max_seq_length = 1024

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading model: {model_name}")
print("This may take a few minutes...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right",
)
tokenizer.pad_token = tokenizer.eos_token

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")
print(f"Model device: {model.device}")

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

## 6. LoRA Configuration

Configure LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,  # rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_ratio = 100 * trainable_params / total_params

print("LoRA configuration applied!")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable ratio: {trainable_ratio:.2f}%")

# Print LoRA modules
print("\nLoRA modules:")
for name, module in model.named_modules():
    if "lora" in name.lower():
        print(f"  {name}")

## 7. Training Loop

Configure and run the training process using the SFTTrainer.

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./vulnerability_model_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    save_strategy="steps",
    evaluation_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    bf16=False,  # Disable bf16 for stability with 4-bit
    report_to="none",  # Disable wandb/tensorboard logging
    max_grad_norm=0.3,
    dataloader_pin_memory=False,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Not masked language modeling
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    max_seq_length=max_seq_length,
    dataset_text_field="text",
)

print("Starting training...")
print(f"Training for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")

# Train the model
trainer.train()

print("Training completed!")

# Save the model
trainer.save_model("./vulnerability_model_final")
print("Model saved to ./vulnerability_model_final")

## 8. Evaluation

Evaluate the trained model on the validation set.

In [ ]:
import re

def extract_predicted_output(generated_text, input_text):
    """Extract the predicted output from generated text."""
    # Remove the input part to get just the generated output
    if input_text in generated_text:
        output_part = generated_text.split(input_text)[-1].strip()
    else:
        output_part = generated_text.strip()

    # Try to extract just the answer part (before any extra text)
    lines = output_part.split('\n')
    for line in lines:
        line = line.strip()
        if line and not line.startswith('Output:'):
            # Look for common answer patterns
            if any(keyword in line.lower() for keyword in ['sql_injection', 'xss', 'path_traversal', 'command_injection', 'safe', 'normalize', 'html_escape', 'parameterized']):
                return line

    # Fallback: return the first non-empty line
    for line in lines:
        if line.strip():
            return line.strip()

    return output_part[:100]  # Fallback to first 100 chars

def evaluate_model(model, tokenizer, eval_dataset, num_samples=20):
    """Evaluate model on a subset of the validation set."""
    model.eval()
    correct = 0
    total = min(num_samples, len(eval_dataset))

    print(f"Evaluating on {total} samples...")

    for i in range(total):
        example = eval_dataset[i]
        input_text = example['text']
        expected_output = example['output']

        # Tokenize input
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_seq_length)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract predicted output
        predicted_output = extract_predicted_output(generated_text, input_text)

        # Check if prediction is correct (simple string matching)
        is_correct = expected_output.lower().strip() in predicted_output.lower().strip() or \
                    predicted_output.lower().strip() in expected_output.lower().strip()

        if is_correct:
            correct += 1

        if i < 5:  # Show first 5 examples
            print(f"\nExample {i+1}:")
            print(f"Input: {input_text[:200]}...")
            print(f"Expected: {expected_output}")
            print(f"Predicted: {predicted_output}")
            print(f"Correct: {is_correct}")

    accuracy = correct / total * 100
    print(f"\nEvaluation Results:")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total})")

    return accuracy

# Run evaluation
accuracy = evaluate_model(model, tokenizer, val_dataset)

## 9. Inference Demo

Demonstrate the trained model on a few example security vulnerability cases.

In [ ]:
def run_inference_demo(model, tokenizer):
    """Run inference on a few example cases."""
    test_cases = [
        {
            "instruction": "Classify this vulnerability:",
            "input": "cursor.execute(f\"UPDATE users SET name='{name}' WHERE id={uid}\")",
            "expected": "sql_injection"
        },
        {
            "instruction": "What is the primary vulnerability in this code?",
            "input": "element.innerHTML = `<img src='${url}'>`;",
            "expected": "xss"
        },
        {
            "instruction": "Choose the correct patch for the path_traversal vulnerability:",
            "input": "Code: open('/data/' + file_path)",
            "options": ["parameterized_query", "normalize_and_allowlist_path", "avoid_shell"],
            "expected": "normalize_and_allowlist_path"
        },
        {
            "instruction": "Is this code vulnerable?",
            "input": "subprocess.run(['ls', '-l', path])",
            "expected": "safe"
        },
        {
            "instruction": "Classify this vulnerability:",
            "input": "os.system(f'curl {url}')",
            "expected": "command_injection"
        }
    ]

    print("🛡️  Vulnerability Classification Model Demo")
    print("=" * 60)

    for i, test_case in enumerate(test_cases, 1):
        print(f"\n🔍 Test Case {i}:")
        print("-" * 40)

        # Format the input
        input_text = f"Instruction: {test_case['instruction']}\nInput: {test_case['input']}"
        if 'options' in test_case:
            input_text += f"\nOptions: {', '.join(test_case['options'])}"
        input_text += "\nOutput:"

        print(f"Input: {test_case['input']}")
        print(f"Expected: {test_case['expected']}")

        # Tokenize and generate
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_seq_length)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode and extract prediction
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predicted_output = extract_predicted_output(generated_text, input_text)

        print(f"Predicted: {predicted_output}")

        # Check accuracy
        expected = test_case['expected'].lower().strip()
        predicted = predicted_output.lower().strip()
        is_correct = expected in predicted or predicted in expected
        print(f"✅ Correct: {is_correct}")

    print("\n" + "=" * 60)
    print("🎉 Demo completed! Your model can classify security vulnerabilities.")

# Run the inference demo
run_inference_demo(model, tokenizer)

## 10. Save and Export Model

Save the trained LoRA adapter and tokenizer for future use.

In [ ]:
# Save the LoRA adapter and tokenizer
output_dir = "./vulnerability_model_final"

print(f"Saving model to {output_dir}...")

# Save the LoRA adapter
model.save_pretrained(output_dir)

# Save the tokenizer
tokenizer.save_pretrained(f"{output_dir}/tokenizer")

print("✅ Model and tokenizer saved successfully!")
print(f"📁 Model location: {output_dir}")
print(f"📁 Adapter: {output_dir}/adapter_model.safetensors")
print(f"📁 Tokenizer: {output_dir}/tokenizer/")

# Create a simple inference script
inference_script = '''
#!/usr/bin/env python3
"""
Inference script for the trained vulnerability classification model.
"""

import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

def load_model(model_path="./vulnerability_model_final"):
    """Load the trained model and tokenizer."""
    print(f"Loading model from {model_path}...")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        f"{model_path}/tokenizer",
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token

    # Load model
    model = AutoPeftModelForCausalLM.from_pretrained(
        f"{model_path}/adapter",
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    return model, tokenizer

def classify_vulnerability(model, tokenizer, instruction, code_snippet, options=None):
    """Classify a vulnerability using the trained model."""
    # Format input
    input_text = f"Instruction: {instruction}\\nInput: {code_snippet}"
    if options:
        input_text += f"\\nOptions: {', '.join(options)}"
    input_text += "\\nOutput:"

    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract prediction (simple extraction - you may want to improve this)
    prediction = generated_text.split("Output:")[-1].strip().split("\\n")[0]

    return prediction

if __name__ == "__main__":
    # Example usage
    model, tokenizer = load_model()

    # Test example
    result = classify_vulnerability(
        model, tokenizer,
        "Classify this vulnerability:",
        "cursor.execute(f\\"UPDATE users SET name='{name}' WHERE id={uid}\\")"
    )

    print(f"Prediction: {result}")
'''

with open("inference_example.py", "w") as f:
    f.write(inference_script)

print("📄 Created inference_example.py for model usage")
print("\n🚀 To use your trained model:")
print("1. Run: python inference_example.py")
print("2. Or load the model in your own code using the load_model() function")
print("\n🎯 Your vulnerability classification model is ready!")